#### Classification du Cancer du Sein à partir d'Images Échographiques

##### Fusion de probabilités DenseNet121 / ResNet50 / InceptionV3

Pipeline de deep learning combinant les prédictions (probabilités softmax) de trois réseaux de neurones
convolutifs pré-entraînés (DenseNet121, ResNet50, InceptionV3) par fusion de caractéristiques au niveau
des probabilités, afin d'améliorer la précision diagnostique sur des images échographiques mammaires
(bénin / malin / normal, dataset BUSI).

**Portée :** expérience académique de machine learning sur un jeu de données public. Ce système n'est
pas un dispositif de diagnostic clinique et n'a subi aucune validation clinique.

#### 1. Imports et configuration

In [4]:
import os
import re
import json
import time
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

CONFIG = {
    "DATA_DIR": "Dataset_BUSI_with_GT",
    "CHECKPOINT_DIR": "checkpoints",
    "RESULTS_DIR": "results",
    "SEED": 42,
    "CLASSES": ["benign", "malignant", "normal"],
    "TRAIN_RATIO": 0.70, "VAL_RATIO": 0.15, "TEST_RATIO": 0.15,
    "IMAGENET_MEAN": [0.485, 0.456, 0.406],
    "IMAGENET_STD": [0.229, 0.224, 0.225],
    "BATCH_SIZE": 32,
    "NUM_WORKERS": 0,
    "WEIGHT_DECAY": 1e-4,
    "GRAD_CLIP_NORM": 1.0,
    "LOSS_TYPE": "focal",
    "LABEL_SMOOTHING": 0.05,
    "FREEZE_BN_STATS_STAGE2": True,
    "MONITOR_METRIC": "f1_macro",
    "USE_ROI_CROP": True,
    "ROI_MARGIN_RATIO": 0.25,
    "USE_TTA": True,
    "TTA_VARIANTS": ["original", "hflip"],   # reduced from 4 -> 2 to cut eval cost
    "INFERENCE_TIMING_WARMUP": 3,
    "INFERENCE_TIMING_REPEATS": 15,
    "USE_AMP": True,
}

os.makedirs(CONFIG["CHECKPOINT_DIR"], exist_ok=True)
os.makedirs(CONFIG["RESULTS_DIR"], exist_ok=True)

CLASS_TO_IDX = {c: i for i, c in enumerate(CONFIG["CLASSES"])}


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG["SEED"])
cuda_available = torch.cuda.is_available()
device = torch.device("cuda" if cuda_available else "cpu")
USE_AMP = CONFIG["USE_AMP"] and cuda_available
print(f"Device: {device} | AMP: {USE_AMP}")

Device: cuda | AMP: True


#### 2. Per-model hyperparameters

Each backbone gets its own settings. DenseNet121 typically converges fastest and cleanest on BUSI, so it
keeps a standard schedule. ResNet50 tends to need a slightly larger backbone LR and more epochs to move
off an early benign/normal-biased optimum on this small, imbalanced dataset. InceptionV3 keeps the v3
fixes (raised backbone LR, lowered aux-loss weight, lowered focal gamma) that resolved its class collapse.

In [5]:
MODEL_SPECS = {
    "densenet121": {
        "img_size": 224, "head_attrs": ["classifier"], "is_inception": False,
        "stage1_epochs": 6, "stage2_epochs": 20, "patience": 6,
        "head_lr": 1e-3, "finetune_head_lr": 1e-4, "finetune_backbone_lr": 1e-5,
        "focal_gamma": 2.0, "aux_loss_weight": 0.0,
    },
    "resnet50": {
        "img_size": 224, "head_attrs": ["fc"], "is_inception": False,
        "stage1_epochs": 8, "stage2_epochs": 26, "patience": 8,
        "head_lr": 1e-3, "finetune_head_lr": 1e-4, "finetune_backbone_lr": 2e-5,
        "focal_gamma": 1.5, "aux_loss_weight": 0.0,
    },
    "inception_v3": {
        "img_size": 299, "head_attrs": ["fc", "AuxLogits"], "is_inception": True,
        "stage1_epochs": 6, "stage2_epochs": 24, "patience": 8,
        "head_lr": 1e-3, "finetune_head_lr": 1e-4, "finetune_backbone_lr": 6e-6,
        "focal_gamma": 1.0, "aux_loss_weight": 0.2,
    },
}

def build_densenet121(num_classes, pretrained=True):
    m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
    m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    return m


def build_resnet50(num_classes, pretrained=True):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m


def build_inception_v3(num_classes, pretrained=True):
    m = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1 if pretrained else None,
                             aux_logits=True, init_weights=False)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    if m.AuxLogits is not None:
        m.AuxLogits.fc = nn.Linear(m.AuxLogits.fc.in_features, num_classes)
    return m


BUILDERS = {"densenet121": build_densenet121, "resnet50": build_resnet50, "inception_v3": build_inception_v3}

#### 3. Dataset loading, mask exclusion, and ROI cropping

In [6]:
data_dir = Path(CONFIG["DATA_DIR"])
if not data_dir.exists():
    raise FileNotFoundError(f"Dataset directory '{data_dir}' not found (expects benign/malignant/normal subfolders).")

IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
MASK_RE = re.compile(r"^(.*)_mask(?:_\d+)?$", re.IGNORECASE)


def mask_base_stem(stem):
    m = MASK_RE.match(stem)
    return m.group(1) if m else stem.lower().split("_mask")[0]


rows, mask_lookup = [], defaultdict(list)
for c in CONFIG["CLASSES"]:
    class_dir = data_dir / c
    images, masks = [], []
    for f in sorted(class_dir.iterdir()):
        if not f.is_file() or f.suffix.lower() not in IMG_EXT:
            continue
        try:
            with Image.open(f) as im:
                im.verify()
        except (UnidentifiedImageError, OSError):
            continue
        (masks if "mask" in f.name.lower() else images).append(f)
    for m in masks:
        mask_lookup[(c, mask_base_stem(m.stem))].append(m)
    for img_path in images:
        rows.append({"filepath": str(img_path), "filename": img_path.name, "class": c,
                     "label": CLASS_TO_IDX[c],
                     "mask_paths": [str(p) for p in mask_lookup.get((c, img_path.stem), [])]})

df = pd.DataFrame(rows)
if len(df) == 0:
    raise RuntimeError("No valid ultrasound images found.")
print(f"Loaded {len(df)} images: {df['class'].value_counts().reindex(CONFIG['CLASSES']).to_dict()}")


def union_bbox_from_masks(mask_paths, threshold=10):
    combined = None
    for mp in mask_paths:
        with Image.open(mp) as m:
            arr = np.array(m.convert("L"))
        binary = arr > threshold
        if not binary.any():
            continue
        ys, xs = np.where(binary)
        box = [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1]
        combined = box if combined is None else [min(combined[0], box[0]), min(combined[1], box[1]),
                                                   max(combined[2], box[2]), max(combined[3], box[3])]
    return tuple(combined) if combined is not None else None


def expand_box(box, w, h, margin_ratio, min_size=16):
    left, top, right, bottom = box
    bw, bh = right - left, bottom - top
    mx, my = margin_ratio * bw, margin_ratio * bh
    l2, t2 = max(0, left - mx), max(0, top - my)
    r2, b2 = min(w, right + mx), min(h, bottom + my)
    if (r2 - l2) < min_size or (b2 - t2) < min_size:
        return None
    return (int(l2), int(t2), int(r2), int(b2))


def matched_center_crop(w, h, target_ratio, rng, min_size=16):
    target_ratio = float(np.clip(target_ratio, 0.1, 0.95))
    side = np.sqrt(target_ratio)
    cw, ch = w * side, h * side
    cx, cy = w / 2 + rng.uniform(-0.05, 0.05) * w, h / 2 + rng.uniform(-0.05, 0.05) * h
    l, t = max(0, cx - cw / 2), max(0, cy - ch / 2)
    r, b = min(w, cx + cw / 2), min(h, cy + ch / 2)
    if (r - l) < min_size or (b - t) < min_size:
        return (0, 0, w, h)
    return (int(l), int(t), int(r), int(b))


lesion_boxes = []
for _, row in df.iterrows():
    box = None
    if row["mask_paths"]:
        bbox = union_bbox_from_masks(row["mask_paths"])
        if bbox is not None:
            with Image.open(row["filepath"]) as im:
                w, h = im.size
            box = expand_box(bbox, w, h, CONFIG["ROI_MARGIN_RATIO"])
    lesion_boxes.append(box)
df["lesion_roi_box"] = lesion_boxes

area_ratios = []
for _, row in df[df["lesion_roi_box"].notna()].iterrows():
    with Image.open(row["filepath"]) as im:
        w, h = im.size
    l, t, r, b = row["lesion_roi_box"]
    area_ratios.append(((r - l) * (b - t)) / (w * h))
area_ratios = np.array(area_ratios) if area_ratios else np.array([0.5])

rng = np.random.RandomState(CONFIG["SEED"])
final_boxes = []
for _, row in df.iterrows():
    if row["lesion_roi_box"] is not None:
        final_boxes.append(row["lesion_roi_box"])
    else:
        with Image.open(row["filepath"]) as im:
            w, h = im.size
        final_boxes.append(matched_center_crop(w, h, rng.choice(area_ratios), rng))
df["roi_box"] = final_boxes
print(f"ROI boxes: {df['lesion_roi_box'].notna().sum()} lesion-guided, "
      f"{df['lesion_roi_box'].isna().sum()} matched-scale fallback")

Loaded 780 images: {'benign': 437, 'malignant': 210, 'normal': 133}
ROI boxes: 647 lesion-guided, 133 matched-scale fallback


#### 4. Train / validation / test split

In [7]:
train_val_df, test_df = train_test_split(df, test_size=CONFIG["TEST_RATIO"], stratify=df["label"],
                                          random_state=CONFIG["SEED"])
val_rel = CONFIG["VAL_RATIO"] / (CONFIG["TRAIN_RATIO"] + CONFIG["VAL_RATIO"])
train_df, val_df = train_test_split(train_val_df, test_size=val_rel, stratify=train_val_df["label"],
                                     random_state=CONFIG["SEED"])
train_df, val_df, test_df = (d.reset_index(drop=True) for d in (train_df, val_df, test_df))
print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

train_counts = train_df["class"].value_counts().reindex(CONFIG["CLASSES"])
n = len(train_df)
class_weights = np.array([n / (3 * train_counts[c]) for c in CONFIG["CLASSES"]], dtype=np.float32)
class_weights = class_weights / class_weights.mean()
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights (focal alpha):", class_weights)

Train: 545  Val: 118  Test: 117
Class weights (focal alpha): [0.47211382 0.9795559  1.5483302 ]


#### 5. Datasets, transforms, and loaders

In [8]:
class BUSIDataset(Dataset):
    def __init__(self, dataframe, transform=None, use_roi_crop=False):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.use_roi_crop = use_roi_crop

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.use_roi_crop and row.get("roi_box") is not None:
            image = image.crop(row["roi_box"])
        if self.transform:
            image = self.transform(image)
        return image, row["label"], row["filename"]


def make_transforms(img_size):
    mean, std = CONFIG["IMAGENET_MEAN"], CONFIG["IMAGENET_STD"]
    train_t = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(), transforms.Normalize(mean, std),
    ])
    eval_t = transforms.Compose([transforms.Resize((img_size, img_size)),
                                  transforms.ToTensor(), transforms.Normalize(mean, std)])
    return train_t, eval_t


def make_tta_transforms(img_size):
    mean, std = CONFIG["IMAGENET_MEAN"], CONFIG["IMAGENET_STD"]
    base = [transforms.Resize((img_size, img_size))]
    tail = [transforms.ToTensor(), transforms.Normalize(mean, std)]
    all_variants = {
        "original": transforms.Compose(base + tail),
        "hflip": transforms.Compose(base + [transforms.RandomHorizontalFlip(1.0)] + tail),
    }
    return {k: all_variants[k] for k in CONFIG["TTA_VARIANTS"]}


def make_loaders(img_size, batch_size, use_roi_crop):
    train_t, eval_t = make_transforms(img_size)
    train_ds = BUSIDataset(train_df, train_t, use_roi_crop)
    val_ds = BUSIDataset(val_df, eval_t, use_roi_crop)
    test_ds = BUSIDataset(test_df, eval_t, use_roi_crop)
    g = torch.Generator(); g.manual_seed(CONFIG["SEED"])
    kw = dict(num_workers=CONFIG["NUM_WORKERS"], pin_memory=cuda_available)
    return (DataLoader(train_ds, batch_size, shuffle=True, generator=g, drop_last=True, **kw),
            DataLoader(val_ds, batch_size, shuffle=False, **kw),
            DataLoader(test_ds, batch_size, shuffle=False, **kw))


USE_ROI_CROP = CONFIG["USE_ROI_CROP"]
LOADERS = {size: make_loaders(size, CONFIG["BATCH_SIZE"], USE_ROI_CROP) for size in {224, 299}}

#### 6. Training utilities

In [9]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha, self.gamma, self.label_smoothing = alpha, gamma, label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none",
                              label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


def get_criterion(gamma):
    if CONFIG["LOSS_TYPE"] == "focal":
        return FocalLoss(alpha=class_weights_tensor, gamma=gamma, label_smoothing=CONFIG["LABEL_SMOOTHING"])
    return nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=CONFIG["LABEL_SMOOTHING"])


def freeze_backbone(model, head_attrs):
    head_ids = {id(p) for name in head_attrs if getattr(model, name, None) is not None
                for p in getattr(model, name).parameters()}
    for p in model.parameters():
        p.requires_grad = id(p) in head_ids


def split_head_backbone(model, head_attrs):
    head_ids, head_params = set(), []
    for name in head_attrs:
        mod = getattr(model, name, None)
        if mod is not None:
            for p in mod.parameters():
                if id(p) not in head_ids:
                    head_ids.add(id(p)); head_params.append(p)
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    return head_params, backbone_params


def freeze_bn_stats(model):
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.eval()


def train_one_epoch(model, loader, optimizer, scheduler, criterion, is_inception, scaler,
                     grad_clip, aux_weight, freeze_bn):
    model.train()
    if freeze_bn:
        freeze_bn_stats(model)
    for images, labels, _ in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=scaler is not None):
            if is_inception:
                out, aux = model(images)
                loss = criterion(out, labels) + aux_weight * criterion(aux, labels)
            else:
                out = model(images)
                loss = criterion(out, labels)
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        if scheduler is not None:
            scheduler.step()


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    all_labels, all_preds = [], []
    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        all_labels.append(labels.cpu().numpy()); all_preds.append(preds.cpu().numpy())
    all_labels, all_preds = np.concatenate(all_labels), np.concatenate(all_preds)
    acc = accuracy_score(all_labels, all_preds)
    _, _, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    return acc, f1


def train_model(model_name, train_loader, val_loader):
    spec = MODEL_SPECS[model_name]
    model = BUILDERS[model_name](num_classes=3).to(device)
    criterion = get_criterion(spec["focal_gamma"])
    ckpt_path = os.path.join(CONFIG["CHECKPOINT_DIR"], f"best_{model_name}.pth")
    best_score, global_epoch = None, 0
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP) if USE_AMP else None

    for stage_name, n_epochs, freeze in [("stage1_head", spec["stage1_epochs"], True),
                                          ("stage2_finetune", spec["stage2_epochs"], False)]:
        if n_epochs <= 0:
            continue
        no_improve = 0
        freeze_bn_stage = (not freeze) and CONFIG["FREEZE_BN_STATS_STAGE2"]
        if freeze:
            freeze_backbone(model, spec["head_attrs"])
            params = [p for p in model.parameters() if p.requires_grad]
            optimizer = optim.AdamW(params, lr=spec["head_lr"], weight_decay=CONFIG["WEIGHT_DECAY"])
            max_lrs = spec["head_lr"]
        else:
            for p in model.parameters():
                p.requires_grad = True
            head_p, backbone_p = split_head_backbone(model, spec["head_attrs"])
            optimizer = optim.AdamW([
                {"params": head_p, "lr": spec["finetune_head_lr"]},
                {"params": backbone_p, "lr": spec["finetune_backbone_lr"]},
            ], weight_decay=CONFIG["WEIGHT_DECAY"])
            max_lrs = [spec["finetune_head_lr"], spec["finetune_backbone_lr"]]

        scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=max_lrs,
                                                   total_steps=len(train_loader) * n_epochs,
                                                   pct_start=0.3, div_factor=10.0, final_div_factor=100.0)
        print(f"\n-- {model_name}: {stage_name} ({n_epochs} epochs, freeze_bn={freeze_bn_stage}) --")
        for _ in range(n_epochs):
            global_epoch += 1
            t0 = time.time()
            train_one_epoch(model, train_loader, optimizer, scheduler, criterion, spec["is_inception"],
                             scaler, CONFIG["GRAD_CLIP_NORM"], spec["aux_loss_weight"], freeze_bn_stage)
            val_acc, val_f1 = validate(model, val_loader, criterion)
            improved = best_score is None or val_f1 > best_score + 1e-4
            if improved:
                best_score = val_f1; no_improve = 0
                torch.save({"model_state_dict": model.state_dict()}, ckpt_path)
            else:
                no_improve += 1
            print(f"epoch {global_epoch:03d} | val_acc {val_acc:.4f} f1_macro {val_f1:.4f} "
                  f"| {time.time()-t0:.1f}s{' *saved*' if improved else ''}")
            if no_improve >= spec["patience"]:
                print(f"Early stopping ({stage_name}).")
                break

    if not os.path.exists(ckpt_path):
        raise RuntimeError(f"No checkpoint saved for {model_name}.")
    print(f"Best val f1_macro for {model_name}: {best_score:.4f}")
    return ckpt_path


def load_best_model(model_name):
    ckpt_path = os.path.join(CONFIG["CHECKPOINT_DIR"], f"best_{model_name}.pth")
    model = BUILDERS[model_name](num_classes=3, pretrained=False).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device)["model_state_dict"])
    model.eval()
    return model


@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    all_probs, all_labels, all_files = [], [], []
    for images, labels, filenames in loader:
        images = images.to(device)
        probs = torch.softmax(model(images), dim=1).cpu().numpy()
        all_probs.append(probs); all_labels.append(labels.numpy()); all_files.extend(filenames)
    return np.concatenate(all_probs), np.concatenate(all_labels), all_files


def predict_probs_tta(model, dataframe, img_size, use_roi_crop):
    variants = make_tta_transforms(img_size)
    variant_probs, labels_ref, files_ref = [], None, None
    for t in variants.values():
        ds = BUSIDataset(dataframe, t, use_roi_crop)
        loader = DataLoader(ds, CONFIG["BATCH_SIZE"], shuffle=False,
                             num_workers=CONFIG["NUM_WORKERS"], pin_memory=cuda_available)
        probs, labels, files = predict_probs(model, loader)
        variant_probs.append(probs)
        labels_ref, files_ref = labels, files
    return np.mean(variant_probs, axis=0), labels_ref, files_ref


def get_probs_for_split(model, dataframe, loader, img_size, use_roi_crop):
    if CONFIG["USE_TTA"]:
        return predict_probs_tta(model, dataframe, img_size, use_roi_crop)
    return predict_probs(model, loader)


@torch.no_grad()
def measure_inference_time_ms(model, loader, n_warmup, n_repeats):
    model.eval()
    images, _, _ = next(iter(loader))
    images = images.to(device)
    for _ in range(n_warmup):
        _ = model(images)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_repeats):
        _ = model(images)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / n_repeats * 1000.0 / images.size(0)


def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"accuracy": acc, "precision_macro": p, "recall_macro": r, "f1_macro": f1}


print("Training utilities ready.")

Training utilities ready.


#### 7. Train the three backbones

In [10]:
CKPTS = {}
for model_name in MODEL_SPECS:
    train_loader, val_loader, _ = LOADERS[MODEL_SPECS[model_name]["img_size"]]
    CKPTS[model_name] = train_model(model_name, train_loader, val_loader)

print("All three models trained.")

C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_26412\2225204330.py:92: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP) if USE_AMP else None



-- densenet121: stage1_head (6 epochs, freeze_bn=False) --
epoch 001 | val_acc 0.1610 f1_macro 0.1586 | 6.0s *saved*
epoch 002 | val_acc 0.4237 f1_macro 0.4226 | 5.9s *saved*
epoch 003 | val_acc 0.6864 f1_macro 0.6815 | 5.6s *saved*
epoch 004 | val_acc 0.7627 f1_macro 0.7454 | 5.5s *saved*
epoch 005 | val_acc 0.7288 f1_macro 0.7157 | 5.5s
epoch 006 | val_acc 0.7373 f1_macro 0.7341 | 5.8s

-- densenet121: stage2_finetune (20 epochs, freeze_bn=True) --


C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_26412\2225204330.py:69: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch 007 | val_acc 0.7542 f1_macro 0.7394 | 6.7s
epoch 008 | val_acc 0.7881 f1_macro 0.7638 | 6.8s *saved*
epoch 009 | val_acc 0.7712 f1_macro 0.7572 | 6.9s
epoch 010 | val_acc 0.8136 f1_macro 0.8061 | 6.9s *saved*
epoch 011 | val_acc 0.8051 f1_macro 0.7979 | 6.7s
epoch 012 | val_acc 0.7881 f1_macro 0.7851 | 6.9s
epoch 013 | val_acc 0.8305 f1_macro 0.8230 | 6.8s *saved*
epoch 014 | val_acc 0.7712 f1_macro 0.7759 | 6.6s
epoch 015 | val_acc 0.7034 f1_macro 0.7398 | 6.8s
epoch 016 | val_acc 0.8390 f1_macro 0.8431 | 6.9s *saved*
epoch 017 | val_acc 0.7881 f1_macro 0.7984 | 6.6s
epoch 018 | val_acc 0.7712 f1_macro 0.7667 | 6.5s
epoch 019 | val_acc 0.7966 f1_macro 0.8166 | 6.4s
epoch 020 | val_acc 0.8136 f1_macro 0.8292 | 6.3s
epoch 021 | val_acc 0.7881 f1_macro 0.8085 | 6.3s
epoch 022 | val_acc 0.8051 f1_macro 0.8273 | 6.2s
Early stopping (stage2_finetune).
Best val f1_macro for densenet121: 0.8431


C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_26412\2225204330.py:92: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP) if USE_AMP else None



-- resnet50: stage1_head (8 epochs, freeze_bn=False) --
epoch 001 | val_acc 0.3644 f1_macro 0.3270 | 5.5s *saved*
epoch 002 | val_acc 0.5508 f1_macro 0.5305 | 5.4s *saved*
epoch 003 | val_acc 0.7458 f1_macro 0.7157 | 5.5s *saved*
epoch 004 | val_acc 0.7034 f1_macro 0.6579 | 5.7s
epoch 005 | val_acc 0.7119 f1_macro 0.6895 | 7.1s
epoch 006 | val_acc 0.7288 f1_macro 0.7018 | 5.4s
epoch 007 | val_acc 0.7288 f1_macro 0.6987 | 5.4s
epoch 008 | val_acc 0.7288 f1_macro 0.7032 | 5.3s

-- resnet50: stage2_finetune (26 epochs, freeze_bn=True) --


... (2 outputs left out)

C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_26412\2225204330.py:92: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP) if USE_AMP else None



-- inception_v3: stage1_head (6 epochs, freeze_bn=False) --
epoch 001 | val_acc 0.3305 f1_macro 0.3364 | 6.5s *saved*
epoch 002 | val_acc 0.5339 f1_macro 0.5480 | 6.5s *saved*
epoch 003 | val_acc 0.7373 f1_macro 0.7181 | 6.4s *saved*
epoch 004 | val_acc 0.7542 f1_macro 0.7263 | 6.6s *saved*
epoch 005 | val_acc 0.7542 f1_macro 0.7469 | 6.3s *saved*
epoch 006 | val_acc 0.7119 f1_macro 0.7002 | 6.2s

-- inception_v3: stage2_finetune (24 epochs, freeze_bn=True) --


C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_26412\2225204330.py:69: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch 007 | val_acc 0.7034 f1_macro 0.6869 | 7.5s
epoch 008 | val_acc 0.7119 f1_macro 0.6945 | 7.4s
epoch 009 | val_acc 0.7542 f1_macro 0.7447 | 7.5s
epoch 010 | val_acc 0.7458 f1_macro 0.7323 | 7.7s
epoch 011 | val_acc 0.7627 f1_macro 0.7513 | 7.9s *saved*
epoch 012 | val_acc 0.8136 f1_macro 0.7952 | 7.3s *saved*
epoch 013 | val_acc 0.8136 f1_macro 0.8101 | 7.5s *saved*
epoch 014 | val_acc 0.8051 f1_macro 0.8139 | 7.6s *saved*
epoch 015 | val_acc 0.8390 f1_macro 0.8470 | 7.7s *saved*
epoch 016 | val_acc 0.8559 f1_macro 0.8722 | 7.8s *saved*
epoch 017 | val_acc 0.8390 f1_macro 0.8470 | 7.3s
epoch 018 | val_acc 0.7966 f1_macro 0.8169 | 7.4s
epoch 019 | val_acc 0.8305 f1_macro 0.8420 | 7.4s
epoch 020 | val_acc 0.8305 f1_macro 0.8528 | 7.4s
epoch 021 | val_acc 0.7966 f1_macro 0.8169 | 7.2s
epoch 022 | val_acc 0.7797 f1_macro 0.7979 | 7.1s
epoch 023 | val_acc 0.8390 f1_macro 0.8547 | 7.7s
epoch 024 | val_acc 0.7627 f1_macro 0.7976 | 7.3s
Early stopping (stage2_finetune).
Best val f1_macro 

#### 8. Evaluate base models + probability fusion (equal / weighted / stacking)

In [11]:
MODELS = {name: load_best_model(name) for name in MODEL_SPECS}

TEST_PROBS, TEST_METRICS, INF_TIME_MS = {}, {}, {}
labels_ref, files_ref = None, None

for name, spec in MODEL_SPECS.items():
    _, _, test_loader = LOADERS[spec["img_size"]]
    probs, labels, files = get_probs_for_split(MODELS[name], test_df, test_loader, spec["img_size"], USE_ROI_CROP)
    if labels_ref is None:
        labels_ref, files_ref = labels, files
    else:
        assert files == files_ref, "Test-set ordering mismatch between models."
    preds = probs.argmax(axis=1)
    TEST_PROBS[name] = probs
    TEST_METRICS[name] = compute_metrics(labels, preds)
    INF_TIME_MS[name] = measure_inference_time_ms(MODELS[name], test_loader,
                                                   CONFIG["INFERENCE_TIMING_WARMUP"],
                                                   CONFIG["INFERENCE_TIMING_REPEATS"])
    print(f"{name}: acc={TEST_METRICS[name]['accuracy']:.4f} "
          f"f1_macro={TEST_METRICS[name]['f1_macro']:.4f} "
          f"inference={INF_TIME_MS[name]:.2f} ms/img")

# --- Equal-weight fusion ---
fused_equal = sum(TEST_PROBS.values()) / len(TEST_PROBS)
metrics_equal = compute_metrics(labels_ref, fused_equal.argmax(axis=1))

# --- Weighted fusion (grid search on validation set) ---
VAL_PROBS = {}
for name, spec in MODEL_SPECS.items():
    _, val_loader, _ = LOADERS[spec["img_size"]]
    probs, val_labels, val_files = get_probs_for_split(MODELS[name], val_df, val_loader, spec["img_size"], USE_ROI_CROP)
    VAL_PROBS[name] = probs

names = list(MODEL_SPECS.keys())
grid = np.arange(0.0, 1.0 + 1e-9, 0.1)
best_f1, best_w = -1.0, (1 / 3, 1 / 3, 1 / 3)
for w1 in grid:
    for w2 in grid:
        w3 = 1.0 - w1 - w2
        if w3 < -1e-9 or w3 > 1.0 + 1e-9:
            continue
        w3 = max(0.0, w3)
        fused_val = w1 * VAL_PROBS[names[0]] + w2 * VAL_PROBS[names[1]] + w3 * VAL_PROBS[names[2]]
        _, _, f1, _ = precision_recall_fscore_support(val_labels, fused_val.argmax(axis=1),
                                                        average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1, best_w = f1, (float(w1), float(w2), float(w3))
tot = sum(best_w)
best_w = tuple(w / tot for w in best_w)
print(f"Best fusion weights on validation ({dict(zip(names, best_w))}), val f1_macro={best_f1:.4f}")

fused_weighted = sum(w * TEST_PROBS[n] for w, n in zip(best_w, names))
metrics_weighted = compute_metrics(labels_ref, fused_weighted.argmax(axis=1))

# --- Stacking meta-learner (logistic regression on concatenated probabilities) ---
X_val = np.concatenate([VAL_PROBS[n] for n in names], axis=1)
X_test = np.concatenate([TEST_PROBS[n] for n in names], axis=1)
meta = LogisticRegression(max_iter=2000, random_state=CONFIG["SEED"])
meta.fit(X_val, val_labels)
metrics_stacking = compute_metrics(labels_ref, meta.predict(X_test))

fusion_inf_time_ms = sum(INF_TIME_MS.values())
print(f"Fusion equal    : acc={metrics_equal['accuracy']:.4f} f1={metrics_equal['f1_macro']:.4f}")
print(f"Fusion weighted : acc={metrics_weighted['accuracy']:.4f} f1={metrics_weighted['f1_macro']:.4f}")
print(f"Fusion stacking : acc={metrics_stacking['accuracy']:.4f} f1={metrics_stacking['f1_macro']:.4f}")

densenet121: acc=0.7607 f1_macro=0.7604 inference=1.43 ms/img
resnet50: acc=0.8547 f1_macro=0.8530 inference=1.28 ms/img
inception_v3: acc=0.5726 f1_macro=0.4091 inference=1.52 ms/img
Best fusion weights on validation ({'densenet121': 0.2, 'resnet50': 0.5, 'inception_v3': 0.30000000000000004}), val f1_macro=0.9428
Fusion equal    : acc=0.8632 f1=0.8577
Fusion weighted : acc=0.8632 f1=0.8628
Fusion stacking : acc=0.8974 f1=0.8939


#### 9. Export — `results/simple_metrics.json` (only report file produced)

In [12]:
def to_simple(m, inf_ms):
    return {"accuracy": round(m["accuracy"], 4), "precision": round(m["precision_macro"], 4),
            "recall": round(m["recall_macro"], 4), "f1_score": round(m["f1_macro"], 4),
            "inference_time_ms_per_image": round(inf_ms, 3)}


simple_results = {
    "densenet121": to_simple(TEST_METRICS["densenet121"], INF_TIME_MS["densenet121"]),
    "resnet50": to_simple(TEST_METRICS["resnet50"], INF_TIME_MS["resnet50"]),
    "inception_v3": to_simple(TEST_METRICS["inception_v3"], INF_TIME_MS["inception_v3"]),
    "fusion_equal_weight": to_simple(metrics_equal, fusion_inf_time_ms),
    "fusion_weighted": to_simple(metrics_weighted, fusion_inf_time_ms),
    "fusion_stacking": to_simple(metrics_stacking, fusion_inf_time_ms),
}

with open(os.path.join(CONFIG["RESULTS_DIR"], "simple_metrics.json"), "w") as f:
    json.dump(simple_results, f, indent=2)

print(json.dumps(simple_results, indent=2))
print("\nSaved: results/simple_metrics.json (only result artifact produced by this script)")

{
  "densenet121": {
    "accuracy": 0.7607,
    "precision": 0.745,
    "recall": 0.8301,
    "f1_score": 0.7604,
    "inference_time_ms_per_image": 1.43
  },
  "resnet50": {
    "accuracy": 0.8547,
    "precision": 0.8311,
    "recall": 0.8913,
    "f1_score": 0.853,
    "inference_time_ms_per_image": 1.275
  },
  "inception_v3": {
    "accuracy": 0.5726,
    "precision": 0.3501,
    "recall": 0.5126,
    "f1_score": 0.4091,
    "inference_time_ms_per_image": 1.516
  },
  "fusion_equal_weight": {
    "accuracy": 0.8632,
    "precision": 0.835,
    "recall": 0.8964,
    "f1_score": 0.8577,
    "inference_time_ms_per_image": 4.222
  },
  "fusion_weighted": {
    "accuracy": 0.8632,
    "precision": 0.8424,
    "recall": 0.8964,
    "f1_score": 0.8628,
    "inference_time_ms_per_image": 4.222
  },
  "fusion_stacking": {
    "accuracy": 0.8974,
    "precision": 0.8768,
    "recall": 0.9166,
    "f1_score": 0.8939,
    "inference_time_ms_per_image": 4.222
  }
}

Saved: results/simple_metr